# Data cleaning <br>

De command hieronder kan gebruikt worden om de data vanuit een bestand te uploaden naar de database. Dit doe je in XAMPP shell.

mysql --local-infile=1 -u root climate_watch

LOAD DATA LOCAL INFILE 'C:/Users/Gebruiker/Documents/case-climate-watch/ClimateData/GlobalTemperatures.csv'
INTO TABLE GlobalTemperatures
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ','
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\r\n'
IGNORE 1 LINES
(
    @dt,
    @LandAverageTemperature,
    @LandAverageTemperatureUncertainty,
    @LandMaxTemperature,
    @LandMaxTemperatureUncertainty,
    @LandMinTemperature,
    @LandMinTemperatureUncertainty,
    @LandAndOceanAverageTemperature,
    @LandAndOceanAverageTemperatureUncertainty
)
SET
    dt = NULLIF(@dt, ''),
    LandAverageTemperature = NULLIF(@LandAverageTemperature, ''),
    LandAverageTemperatureUncertainty = NULLIF(@LandAverageTemperatureUncertainty, ''),
    LandMaxTemperature = NULLIF(@LandMaxTemperature, ''),
    LandMaxTemperatureUncertainty = NULLIF(@LandMaxTemperatureUncertainty, ''),
    LandMinTemperature = NULLIF(@LandMinTemperature, ''),
    LandMinTemperatureUncertainty = NULLIF(@LandMinTemperatureUncertainty, ''),
    LandAndOceanAverageTemperature = NULLIF(@LandAndOceanAverageTemperature, ''),
    LandAndOceanAverageTemperatureUncertainty = NULLIF(@LandAndOceanAverageTemperatureUncertainty, '');

In [20]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

engine = create_engine(
    "mysql+pymysql://root@localhost:3306/climate_watch"
)

df_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCity", con=engine, parse_dates=["dt"])
df_country = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCountry", con=engine, parse_dates=["dt"])
df_major_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByMajorCity", con=engine, parse_dates=["dt"])
df_state = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByState", con=engine, parse_dates=["dt"])
df_global = pd.read_sql("SELECT * FROM GlobalTemperatures", con=engine, parse_dates=["dt"])

## Often used functions

In [29]:
def decompose_date(df: pd.DataFrame) -> pd.DataFrame:
    df["year"] = df["dt"].dt.year
    df["month"] = df["dt"].dt.month
    df["day"] = df["dt"].dt.day
    return df

def fill_na_with_avg(df: pd.DataFrame, fill_column: str, group_column: list[str], sort_columns: list[str]) -> pd.DataFrame:
    """
    First, sort the dataframe by 'sort_columns'.
    Then replace the missing values of 'fill_column' by the mean value of the previous en next value of that 'group_column'.
    Note that this will not work if there are 3 consecutive missing values.
    """
    df = df.sort_values(sort_columns)
    df["previous"] = df.groupby(group_column)[fill_column].shift(1)
    df["next"] = df.groupby(group_column)[fill_column].shift(-1)
    df["average"] = df[["previous", "next"]].mean(skipna=False, axis=1)     # if either value is NaN, "average" will get NaN
    df[fill_column] = df[fill_column].fillna(df["average"])
    df = df.drop(columns=["previous", "next", "average"])

    return df

def update_data_in_db(df: pd.DataFrame, table_name: str) -> None:
    """
    To keep the table structure, I will first delete all data and then append the current data to the table.
    """
    with engine.begin() as connection:
        connection.execute(text(f"TRUNCATE TABLE {table_name}"))
        df.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False,
            chunksize=5000
            )

## City data

In [23]:
for col in df_city.columns:
    print(f"{col}: {"Missing values" if df_city[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per city:")
df_city[df_city.isna().any(axis=1)].City.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
City: No missing values
Country: No missing values
Latitude: No missing values
Longitude: No missing values


Nr of missing values per city:


City
Port Louis      1039
Saint Denis     1039
Fianarantsoa    1036
Antananarivo     958
Antsirabe        958
                ... 
Zhuji              1
Zhumadian          1
Zouxian            1
Zunyi              1
Zuwarah            1
Name: count, Length: 3323, dtype: int64

There are few options for dealing with the missing values:
- Remove any row with missing values
- Replace missing values by 0
- Replace missing value by avg value of that country
- Replace missing value by avg value of that country in the same month and same year
- Replace missing value by avg of previous en next value of that country

For now I have the following cases:
- If there is 1 consecutive missing values, I will replace interpolate them using the surrounding values.
- If there are more than 2 consecutive missing values, I will leave them in for now. The effects will be measured later.

In [30]:
df_city = fill_na_with_avg(df_city, "AverageTemperature", ["City", "Country"], ["City", "Country", "dt"])
df_city = fill_na_with_avg(df_city, "AverageTemperatureUncertainty", ["City", "Country"], ["City", "Country", "dt"])

update_data_in_db(df_city, "GlobalLandTemperaturesByCity")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_22608\3339263641.py:28: UserWarning: The provided table name 'GlobalLandTemperaturesByCity' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Country data

In [36]:
for col in df_country.columns:
    print(f"{col}: {"Missing values" if df_country[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per country:")
display(df_country[df_country.isna().any(axis=1)].Country.value_counts())
df_country_antarctica = df_country[df_country["Country"] == "Antarctica"]
print(f"{len(df_country_antarctica.isna())}/{len(df_country_antarctica)} rows from Antarctica are missing data.")

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
Country: No missing values


Nr of missing values per country:


Country
Mauritius        1039
Reunion          1039
Comoros           958
Madagascar        958
Mayotte           958
                 ... 
South America       1
South Korea         1
Syria               1
Timor Leste         1
Tunisia             1
Name: count, Length: 238, dtype: int64

764/764 rows from Antarctica are missing data.


In [38]:
df_country = df_country[df_country["Country"] != "Antarctica"].reset_index(drop=True)
df_country = fill_na_with_avg(df_country, "AverageTemperature", ["Country"], ["Country", "dt"])
df_country = fill_na_with_avg(df_country, "AverageTemperatureUncertainty", ["Country"], ["Country", "dt"])

update_data_in_db(df_country, "GlobalLandTemperaturesByCountry")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_22608\3339263641.py:28: UserWarning: The provided table name 'GlobalLandTemperaturesByCountry' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Major city data

In [39]:
for col in df_major_city.columns:
    print(f"{col}: {"Missing values" if df_major_city[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per major city:")
df_major_city[df_major_city.isna().any(axis=1)].City.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
City: No missing values
Country: No missing values
Latitude: No missing values
Longitude: No missing values


Nr of missing values per major city:


City
Jakarta          386
Surabaya         386
Dar Es Salaam    379
Fortaleza        366
Karachi          289
                ... 
Nanjing            1
Santiago           1
Seoul              1
Shanghai           1
Wuhan              1
Name: count, Length: 98, dtype: int64

In [40]:
df_major_city = fill_na_with_avg(df_major_city, "AverageTemperature", ["City", "Country"], ["City", "Country", "dt"])
df_major_city = fill_na_with_avg(df_major_city, "AverageTemperatureUncertainty", ["City", "Country"], ["City", "Country", "dt"])

update_data_in_db(df_major_city, "GlobalLandTemperaturesByMajorCity")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_22608\3339263641.py:28: UserWarning: The provided table name 'GlobalLandTemperaturesByMajorCity' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## State data

In [41]:
for col in df_state.columns:
    print(f"{col}: {"Missing values" if df_state[col].isna().any() else "No missing values"}")

print("\n\nNr of missing values per state:")
df_state[df_state.isna().any(axis=1)].State.value_counts()

dt: No missing values
AverageTemperature: Missing values
AverageTemperatureUncertainty: Missing values
State: No missing values
Country: No missing values


Nr of missing values per state:


State
Nunavut                  679
Louisiana                560
Oklahoma                 519
Northwest Territories    493
Alberta                  485
                        ... 
Taymyr                     1
Tuva                       1
Ust Orda Buryat            1
Yevrey                     1
Zhejiang                   1
Name: count, Length: 239, dtype: int64

In [42]:
df_state = fill_na_with_avg(df_state, "AverageTemperature", ["State", "Country"], ["State", "Country", "dt"])
df_state = fill_na_with_avg(df_state, "AverageTemperatureUncertainty", ["State", "Country"], ["State", "Country", "dt"])

update_data_in_db(df_state, "GlobalLandTemperaturesByState")

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_22608\3339263641.py:28: UserWarning: The provided table name 'GlobalLandTemperaturesByState' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(


## Global data

In [ ]:
for col in df_global.columns:
    print(f"{col}: {"Missing values" if df_global[col].isna().any() else "No missing values"}")
    print(f"Nr of missing values: {len(df_global[col].isna())}\n")


dt: No missing values
Nr of missing values: 3192

LandAverageTemperature: Missing values
Nr of missing values: 3192

LandAverageTemperatureUncertainty: Missing values
Nr of missing values: 3192

LandMaxTemperature: Missing values
Nr of missing values: 3192

LandMaxTemperatureUncertainty: Missing values
Nr of missing values: 3192

LandMinTemperature: Missing values
Nr of missing values: 3192

LandMinTemperatureUncertainty: Missing values
Nr of missing values: 3192

LandAndOceanAverageTemperature: Missing values
Nr of missing values: 3192

LandAndOceanAverageTemperatureUncertainty: Missing values
Nr of missing values: 3192

